In [1]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝


import sys
import importlib

def install_if_missing(package_name: str):
    try:
        importlib.import_module(package_name)
    except ImportError:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_name])


for pkg in ["pandas", "plotly", "ipywidgets"]:
    install_if_missing(pkg)


try:
    import ipywidgets as widgets
    from IPython.display import display, Javascript
    widgets.IntSlider()  
    display(Javascript("Jupyter.notebook.kernel.execute('from ipywidgets import widgets')"))
except:
    pass

print("All packages ready! FPMA dashboard will work perfectly.")

<IPython.core.display.Javascript object>

All packages ready! FPMA dashboard will work perfectly.


In [2]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝
!pip install statsmodels

In [3]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import pandas as pd

csv_url = "https://raw.githubusercontent.com/tezamo/FPMA/main/wheat/wheat.csv"

df = pd.read_csv(csv_url, parse_dates=["date"], dayfirst=True)

# Display the first rows
df.head()

HTTPError: HTTP Error 404: Not Found

In [ ]:
'''
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

# Month format checking

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df.head()

'''

In [ ]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝


#==================
# Imports
#==================
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from statsmodels.tsa.stattools import adfuller, acf, pacf
import plotly.express as px
import numpy as np


#==================
# create_widget_set
#==================
def create_widget_set(name_prefix):
    source_dropdown = widgets.Dropdown(options=["Domestic", "International"], description=f"{name_prefix} Source:")
    commodity_type_dropdown = widgets.Dropdown(description=f"{name_prefix} Commodity Type:")
    commodity_dropdown = widgets.Dropdown(description=f"{name_prefix} Commodity:")
    price_type_dropdown = widgets.Dropdown(description=f"{name_prefix} Price Type:")
    country_dropdown = widgets.Dropdown(description=f"{name_prefix} Country:")
    market_dropdown = widgets.Dropdown(description=f"{name_prefix} Market:")

    #==================
    # Function
    #==================
    def update_widgets(*args):
        df_src = df[df["price_source"] == source_dropdown.value]

        # COMMODITY TYPE 
        types = sorted(df_src["commodity_type"].dropna().unique())
        commodity_type_dropdown.options = types
        if commodity_type_dropdown.value not in types:
            commodity_type_dropdown.value = types[0] if types else None

        df_type = df_src[df_src["commodity_type"] == commodity_type_dropdown.value] \
                  if commodity_type_dropdown.value else df_src

        # COMMODITY NAME 
        commodities = sorted(df_type["commodity_name"].dropna().unique())
        commodity_dropdown.options = commodities
        if commodity_dropdown.value not in commodities:
            commodity_dropdown.value = commodities[0] if commodities else None

        df_com = df_type[df_type["commodity_name"] == commodity_dropdown.value] \
                if commodity_dropdown.value else df_type

        # PRICE TYPE
        price_types = sorted(df_com["price_type"].dropna().unique())
        price_type_dropdown.options = price_types
        if price_type_dropdown.value not in price_types:
            price_type_dropdown.value = price_types[0] if price_types else None

        df_pt = df_com[df_com["price_type"] == price_type_dropdown.value] \
                if price_type_dropdown.value else df_com

        # COUNTRY
        countries = sorted(df_pt["country"].dropna().unique())
        country_dropdown.options = ["All"] + countries
        if country_dropdown.value not in countries:
            country_dropdown.value = "All"

        df_ct = df_pt if country_dropdown.value == "All" else df_pt[df_pt["country"] == country_dropdown.value]

        # MARKET
        markets = sorted(df_ct["market"].dropna().unique())
        market_dropdown.options = ["All"] + markets
        if market_dropdown.value not in markets:
            market_dropdown.value = "All"

    source_dropdown.observe(update_widgets, "value")
    commodity_type_dropdown.observe(update_widgets, "value")
    commodity_dropdown.observe(update_widgets, "value")
    price_type_dropdown.observe(update_widgets, "value")
    country_dropdown.observe(update_widgets, "value")

    update_widgets()

    return {
        "source": source_dropdown,
        "commodity_type": commodity_type_dropdown,
        "commodity": commodity_dropdown,
        "price_type": price_type_dropdown,
        "country": country_dropdown,
        "market": market_dropdown
    }

#==================
#  Plot & Descriptive Analysis
#==================
selections = []
output_plot = widgets.Output()

def compute_adf(series):
    """Run Augmented Dickey-Fuller test and return results as a dictionary."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Observations Used": result[3]
    }

def create_acf_df(series, nlags=20):
    """Return ACF values for lags as a DataFrame."""
    values = acf(series.dropna(), nlags=nlags)
    return pd.DataFrame({"Lag": range(len(values)), "ACF": values})

def create_pacf_df(series, nlags=20):
    """Return PACF values for lags as a DataFrame."""
    values = pacf(series.dropna(), nlags=nlags)
    return pd.DataFrame({"Lag": range(len(values)), "PACF": values})

def monthly_seasonality(series, dates):
    """Compute mean value for each month."""
    df_tmp = pd.DataFrame({"value": series, "month": dates.dt.month})
    return df_tmp.groupby("month")["value"].mean().reset_index()

def update_plot():
    output_plot.clear_output()
    with output_plot:
        if not selections:
            display("⚠ No selections.")
            return

        fig = go.Figure()
        any_data = False

        #=====================================================
        # PLOT
        #=====================================================
        for sel in selections:
            dff = df[df["price_source"] == sel["source"]]

            if sel["commodity_type"]:
                dff = dff[dff["commodity_type"] == sel["commodity_type"]]
            if sel["commodity"] != "All":
                dff = dff[dff["commodity_name"] == sel["commodity"]]
            if sel["price_type"] != "All":
                dff = dff[dff["price_type"] == sel["price_type"]]
            if sel["country"] != "All":
                dff = dff[dff["country"] == sel["country"]]
            if sel["market"] != "All":
                dff = dff[dff["market"] == sel["market"]]

            if dff.empty:
                print(f"⚠ No data for {sel['label']}")
                continue

            any_data = True

            dff = dff.copy()
            dff["country"] = dff["country"].fillna("Unknown")
            dff["market"] = dff["market"].fillna("Unknown")

            units = dff["unit_std"].dropna().unique()
            unit_label = units[0] if len(units) == 1 else "Multiple Units"

            # Legend string
            dff["legend_label"] = (
                dff["commodity_name"].astype(str) + " • " +
                unit_label + " • " +
                dff["country"].astype(str) + " • " +
                dff["market"].astype(str) + " • " +
                dff["price_type"].astype(str) + " • " +
                dff["price_source"].astype(str)
            )

            for name, group in dff.groupby("legend_label"):
                group_sorted = group.sort_values("date")
                fig.add_trace(go.Scatter(
                    x=group_sorted["date"],
                    y=group_sorted["price_per_unit"],
                    mode="lines",
                    name=name,
                    hovertemplate="%{x}<br>%{y}<extra></extra>"
                ))

        if not any_data:
            display("⚠ No data available for the selections.")
            return

        fig.update_layout(
            title="Plot",
            xaxis_title="Date",
            yaxis_title="Price per Unit",
            legend_title=""
        )
        display(fig)

        #=====================================================
        # DESCRIPTIVE + TIME-SERIES TABS
        #=====================================================
        tab_titles = []
        tab_contents = []

        for sel in selections:
            dff_s = df[df["price_source"] == sel["source"]]

            if sel["commodity_type"]:
                dff_s = dff_s[dff_s["commodity_type"] == sel["commodity_type"]]
            if sel["commodity"] != "All":
                dff_s = dff_s[dff_s["commodity_name"] == sel["commodity"]]
            if sel["price_type"] != "All":
                dff_s = dff_s[dff_s["price_type"] == sel["price_type"]]
            if sel["country"] != "All":
                dff_s = dff_s[dff_s["country"] == sel["country"]]
            if sel["market"] != "All":
                dff_s = dff_s[dff_s["market"] == sel["market"]]

            if dff_s.empty:
                continue

            # -------- DESCRIPTIVE TAB --------
            out = widgets.Output()
            with out:

                unit_options = dff_s["unit_std"].dropna().unique()
                unit_label_s = unit_options[0] if len(unit_options) == 1 else "Multiple Units"

                stats = {
                    "Count": len(dff_s),
                    "Num Countries": dff_s["country"].nunique(),
                    "Num Series": len(dff_s.groupby(["country", "market", "price_type", "commodity_name"])),
                    "Overall start": dff_s["date"].min(),
                    "Overall end": dff_s["date"].max(),
                    "Min": dff_s["price_per_unit"].min(),
                    "Max": dff_s["price_per_unit"].max(),
                    "Mean": dff_s["price_per_unit"].mean(),
                    "Median": dff_s["price_per_unit"].median(),
                    "Std": dff_s["price_per_unit"].std(),
                    "Range": dff_s["price_per_unit"].max() - dff_s["price_per_unit"].min(),
                    "% Change": (
                        (dff_s["price_per_unit"].iloc[-1] - dff_s["price_per_unit"].iloc[0]) 
                        / dff_s["price_per_unit"].iloc[0] * 100
                    ) if len(dff_s) > 1 else None
                }

                stats_df = pd.DataFrame({"Metric": stats.keys(), "Value": stats.values()})
                country_count = dff_s.groupby("country")["price_per_unit"].count().rename("Count").to_frame()
                start_end = dff_s.groupby("country")["date"].agg(["min", "max"])
                nulls = dff_s.isna().sum().rename("Nulls").to_frame()
                gaps_df = (
                    dff_s.sort_values("date")["date"]
                    .diff().dt.days
                    .value_counts().sort_index()
                    .rename("Occurrences").to_frame()
                )

                display(widgets.HTML(
                    f"<h3>{sel['commodity_type']} • {sel['commodity']} • {unit_label_s} • "
                    f"{sel['country']} • {sel['market']} • {sel['price_type']} • {sel['source']}</h3>"
                ))

                display(widgets.HTML("<b>Descriptive Statistics</b>"))
                display(stats_df)

                display(widgets.HTML('<hr><b>Data Length per Country</b>'))
                display(country_count)

                display(widgets.HTML('<hr><b>Start / End Dates per Country</b>'))
                display(start_end)

                display(widgets.HTML('<hr><b>Null Counts</b>'))
                display(nulls)

                display(widgets.HTML('<hr><b>Frequency Gaps (days)</b>'))
                display(gaps_df)


            # -------- TIME-SERIES TAB --------
            ts_out = widgets.Output()
            with ts_out:
                ts = dff_s.sort_values("date")
                ts_values = ts["price_per_unit"]
                ts_dates = ts["date"]

                # Rolling stats
                ts["rolling_mean"] = ts_values.rolling(window=12).mean()
                ts["rolling_std"] = ts_values.rolling(window=12).std()

                fig_roll = go.Figure()
                fig_roll.add_trace(go.Scatter(x=ts_dates, y=ts_values, mode="lines", name="Price"))
                fig_roll.add_trace(go.Scatter(x=ts_dates, y=ts["rolling_mean"], mode="lines",
                                              name="Rolling Mean (12)", line=dict(dash="dash")))
                fig_roll.add_trace(go.Scatter(x=ts_dates, y=ts["rolling_std"], mode="lines",
                                              name="Rolling Std (12)", line=dict(dash="dot")))
                fig_roll.update_layout(title="Rolling Mean & Std")
                display(fig_roll)

                # ACF / PACF
                display(px.bar(create_acf_df(ts_values), x="Lag", y="ACF", title="ACF"))
                display(px.bar(create_pacf_df(ts_values), x="Lag", y="PACF", title="PACF"))

                # ADF
                adf_res = compute_adf(ts_values)
                adf_df = pd.DataFrame({"Metric": adf_res.keys(), "Value": adf_res.values()})
                display(widgets.HTML("<b>ADF Stationarity Test</b>"))
                display(adf_df)

                # Seasonality
                seas = monthly_seasonality(ts_values, ts_dates)
                display(px.bar(seas, x="month", y="value", title="Monthly Seasonality"))

            # ADD TABS
            tab_titles.append(f"{sel['commodity']} ({sel['country']})")
            tab_contents.append(out)

            tab_titles.append(f"{sel['commodity']} ({sel['country']}) - TS")
            tab_contents.append(ts_out)

        # Show final tabs
        if tab_contents:
            tabs = widgets.Tab(children=tab_contents)
            for i, t in enumerate(tab_titles):
                tabs.set_title(i, t[:25])
            display(tabs)


#==================
# Add / Remove / Clear Buttons
#==================
widget_set = create_widget_set("")
selection_list = selections

btn_add = widgets.Button(description="➕ Add Selection", button_style='success')
btn_remove = widgets.Button(description="❌ Remove Selected", button_style='warning')
btn_clear = widgets.Button(description="🗑 Clear All", button_style='danger')

remove_dropdown = widgets.Dropdown(options=[], description="Remove:", layout=widgets.Layout(width="450px"))

def refresh_remove_dropdown():
    if not selection_list:
        remove_dropdown.options = []
        remove_dropdown.value = None
        return
    labels = [
        f"{i+1}: {s['commodity_type']} | {s['commodity']} | {s['price_type']} | {s['country']} | {s['market']} | {s['source']}"
        for i, s in enumerate(selection_list)
    ]
    remove_dropdown.options = labels
    remove_dropdown.value = labels[0]

def add_selection(_):
    vals = {k: w.value for k, w in widget_set.items()}
    vals["label"] = (
        f"{vals['commodity_type']} | {vals['commodity']} | {vals['price_type']} | "
        f"{vals['source']} | {vals['country']} | {vals['market']}"
    )
    selection_list.append(vals)
    refresh_remove_dropdown()
    update_plot()

def remove_selected(_):
    if not selection_list or not remove_dropdown.value:
        return
    idx = remove_dropdown.options.index(remove_dropdown.value)
    del selection_list[idx]
    refresh_remove_dropdown()
    update_plot()

def clear_all(_):
    selection_list.clear()
    refresh_remove_dropdown()
    update_plot()

btn_add.on_click(add_selection)
btn_remove.on_click(remove_selected)
btn_clear.on_click(clear_all)

#==================
# Display
#==================
ui = widgets.VBox(
    list(widget_set.values()) +
    [widgets.HBox([btn_add, btn_remove, btn_clear]), remove_dropdown, output_plot]
)
display(ui)

In [ ]:
# adding All option to price type and commodity too

# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝


#==================
# Imports
#==================
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from statsmodels.tsa.stattools import adfuller, acf, pacf
import plotly.express as px
import numpy as np


#==================
# create_widget_set
#==================
def create_widget_set(name_prefix):
    source_dropdown = widgets.Dropdown(options=["Domestic", "International"], description=f"{name_prefix} Source:")
    commodity_type_dropdown = widgets.Dropdown(description=f"{name_prefix} Commodity Type:")
    commodity_dropdown = widgets.Dropdown(description=f"{name_prefix} Commodity:")
    price_type_dropdown = widgets.Dropdown(description=f"{name_prefix} Price Type:")
    country_dropdown = widgets.Dropdown(description=f"{name_prefix} Country:")
    market_dropdown = widgets.Dropdown(description=f"{name_prefix} Market:")

    #==================
    # Function
    #==================
    def update_widgets(*args):
        df_src = df[df["price_source"] == source_dropdown.value]

        # COMMODITY TYPE 
        types = sorted(df_src["commodity_type"].dropna().unique())
        commodity_type_dropdown.options = types
        if commodity_type_dropdown.value not in types:
            commodity_type_dropdown.value = types[0] if types else None

        df_type = df_src[df_src["commodity_type"] == commodity_type_dropdown.value] \
                  if commodity_type_dropdown.value else df_src

        
        # COMMODITY NAME 
        commodities = sorted(df_type["commodity_name"].dropna().unique())
        commodity_dropdown.options = ["All"] + commodities
        if commodity_dropdown.value not in commodity_dropdown.options:
           commodity_dropdown.value = "All"


        df_com = (
            df_type
            if commodity_dropdown.value == "All"
             else df_type[df_type["commodity_name"] == commodity_dropdown.value]
        )


        
        # PRICE TYPE
        price_types = sorted(df_com["price_type"].dropna().unique())
        price_type_dropdown.options = ["All"] + price_types
        if price_type_dropdown.value not in price_type_dropdown.options:
           price_type_dropdown.value = "All"


        df_pt = (
         df_com
        if price_type_dropdown.value == "All"
        else df_com[df_com["price_type"] == price_type_dropdown.value]
        )

        # COUNTRY
        countries = sorted(df_pt["country"].dropna().unique())
        country_dropdown.options = ["All"] + countries
        if country_dropdown.value not in countries:
            country_dropdown.value = "All"

        df_ct = df_pt if country_dropdown.value == "All" else df_pt[df_pt["country"] == country_dropdown.value]

        # MARKET
        markets = sorted(df_ct["market"].dropna().unique())
        market_dropdown.options = ["All"] + markets
        if market_dropdown.value not in markets:
            market_dropdown.value = "All"

    source_dropdown.observe(update_widgets, "value")
    commodity_type_dropdown.observe(update_widgets, "value")
    commodity_dropdown.observe(update_widgets, "value")
    price_type_dropdown.observe(update_widgets, "value")
    country_dropdown.observe(update_widgets, "value")

    update_widgets()

    return {
        "source": source_dropdown,
        "commodity_type": commodity_type_dropdown,
        "commodity": commodity_dropdown,
        "price_type": price_type_dropdown,
        "country": country_dropdown,
        "market": market_dropdown
    }

#==================
#  Plot & Descriptive Analysis
#==================
selections = []
output_plot = widgets.Output()

def compute_adf(series):
    """Run Augmented Dickey-Fuller test and return results as a dictionary."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Observations Used": result[3]
    }

def create_acf_df(series, nlags=20):
    """Return ACF values for lags as a DataFrame."""
    values = acf(series.dropna(), nlags=nlags)
    return pd.DataFrame({"Lag": range(len(values)), "ACF": values})

def create_pacf_df(series, nlags=20):
    """Return PACF values for lags as a DataFrame."""
    values = pacf(series.dropna(), nlags=nlags)
    return pd.DataFrame({"Lag": range(len(values)), "PACF": values})

def monthly_seasonality(series, dates):
    """Compute mean value for each month."""
    df_tmp = pd.DataFrame({"value": series, "month": dates.dt.month})
    return df_tmp.groupby("month")["value"].mean().reset_index()

def update_plot():
    output_plot.clear_output()
    with output_plot:
        if not selections:
            display("⚠ No selections.")
            return

        fig = go.Figure()
        any_data = False

        #=====================================================
        # PLOT
        #=====================================================
        for sel in selections:
            dff = df[df["price_source"] == sel["source"]]

            if sel["commodity_type"]:
                dff = dff[dff["commodity_type"] == sel["commodity_type"]]
            if sel["commodity"] != "All":
                dff = dff[dff["commodity_name"] == sel["commodity"]]
            if sel["price_type"] != "All":
                dff = dff[dff["price_type"] == sel["price_type"]]
            if sel["country"] != "All":
                dff = dff[dff["country"] == sel["country"]]
            if sel["market"] != "All":
                dff = dff[dff["market"] == sel["market"]]

            if dff.empty:
                print(f"⚠ No data for {sel['label']}")
                continue

            any_data = True

            dff = dff.copy()
            dff["country"] = dff["country"].fillna("Unknown")
            dff["market"] = dff["market"].fillna("Unknown")

            units = dff["unit_std"].dropna().unique()
            unit_label = units[0] if len(units) == 1 else "Multiple Units"

            # Legend string
            dff["legend_label"] = (
                dff["commodity_name"].astype(str) + " • " +
                unit_label + " • " +
                dff["country"].astype(str) + " • " +
                dff["market"].astype(str) + " • " +
                dff["price_type"].astype(str) + " • " +
                dff["price_source"].astype(str)
            )

            for name, group in dff.groupby("legend_label"):
                group_sorted = group.sort_values("date")
                fig.add_trace(go.Scatter(
                    x=group_sorted["date"],
                    y=group_sorted["price_per_unit"],
                    mode="lines",
                    name=name,
                    hovertemplate="%{x}<br>%{y}<extra></extra>"
                ))

        if not any_data:
            display("⚠ No data available for the selections.")
            return

        fig.update_layout(
            title="Combined Plot",
            xaxis_title="Date",
            yaxis_title="Price per Unit",
            legend_title=""
        )
        fig.show()

        #=====================================================
        # DESCRIPTIVE + TIME-SERIES TABS
        #=====================================================
        tab_titles = []
        tab_contents = []

        for sel in selections:
            dff_s = df[df["price_source"] == sel["source"]]

            if sel["commodity_type"]:
                dff_s = dff_s[dff_s["commodity_type"] == sel["commodity_type"]]
            if sel["commodity"] != "All":
                dff_s = dff_s[dff_s["commodity_name"] == sel["commodity"]]
            if sel["price_type"] != "All":
                dff_s = dff_s[dff_s["price_type"] == sel["price_type"]]
            if sel["country"] != "All":
                dff_s = dff_s[dff_s["country"] == sel["country"]]
            if sel["market"] != "All":
                dff_s = dff_s[dff_s["market"] == sel["market"]]

            if dff_s.empty:
                continue

            # -------- DESCRIPTIVE TAB --------
            out = widgets.Output()
            with out:

                unit_options = dff_s["unit_std"].dropna().unique()
                unit_label_s = unit_options[0] if len(unit_options) == 1 else "Multiple Units"

                stats = {
                    "Count": len(dff_s),
                    "Num Countries": dff_s["country"].nunique(),
                    "Num Series": len(dff_s.groupby(["country", "market", "price_type", "commodity_name"])),
                    "Overall start": dff_s["date"].min(),
                    "Overall end": dff_s["date"].max(),
                    "Min": dff_s["price_per_unit"].min(),
                    "Max": dff_s["price_per_unit"].max(),
                    "Mean": dff_s["price_per_unit"].mean(),
                    "Median": dff_s["price_per_unit"].median(),
                    "Std": dff_s["price_per_unit"].std(),
                    "Range": dff_s["price_per_unit"].max() - dff_s["price_per_unit"].min(),
                    "% Change": (
                        (dff_s["price_per_unit"].iloc[-1] - dff_s["price_per_unit"].iloc[0]) 
                        / dff_s["price_per_unit"].iloc[0] * 100
                    ) if len(dff_s) > 1 else None
                }

                stats_df = pd.DataFrame({"Metric": stats.keys(), "Value": stats.values()})
                country_count = dff_s.groupby("country")["price_per_unit"].count().rename("Count").to_frame()
                start_end = dff_s.groupby("country")["date"].agg(["min", "max"])
                nulls = dff_s.isna().sum().rename("Nulls").to_frame()
                gaps_df = (
                    dff_s.sort_values("date")["date"]
                    .diff().dt.days
                    .value_counts().sort_index()
                    .rename("Occurrences").to_frame()
                )

                display(widgets.HTML(
                    f"<h3>{sel['commodity_type']} • {sel['commodity']} • {unit_label_s} • "
                    f"{sel['country']} • {sel['market']} • {sel['price_type']} • {sel['source']}</h3>"
                ))

                display(widgets.HTML("<b>Descriptive Statistics</b>"))
                display(stats_df)

                display(widgets.HTML('<hr><b>Data Length per Country</b>'))
                display(country_count)

                display(widgets.HTML('<hr><b>Start / End Dates per Country</b>'))
                display(start_end)

                display(widgets.HTML('<hr><b>Null Counts</b>'))
                display(nulls)

                display(widgets.HTML('<hr><b>Frequency Gaps (days)</b>'))
                display(gaps_df)


            # -------- TIME-SERIES TAB --------
            ts_out = widgets.Output()
            with ts_out:
                ts = dff_s.sort_values("date")
                ts_values = ts["price_per_unit"]
                ts_dates = ts["date"]

                # Rolling stats
                ts["rolling_mean"] = ts_values.rolling(window=12).mean()
                ts["rolling_std"] = ts_values.rolling(window=12).std()

                fig_roll = go.Figure()
                fig_roll.add_trace(go.Scatter(x=ts_dates, y=ts_values, mode="lines", name="Price"))
                fig_roll.add_trace(go.Scatter(x=ts_dates, y=ts["rolling_mean"], mode="lines",
                                              name="Rolling Mean (12)", line=dict(dash="dash")))
                fig_roll.add_trace(go.Scatter(x=ts_dates, y=ts["rolling_std"], mode="lines",
                                              name="Rolling Std (12)", line=dict(dash="dot")))
                fig_roll.update_layout(title="Rolling Mean & Std")
                display(fig_roll)

                # ACF / PACF
                display(px.bar(create_acf_df(ts_values), x="Lag", y="ACF", title="ACF"))
                display(px.bar(create_pacf_df(ts_values), x="Lag", y="PACF", title="PACF"))

                # ADF
                adf_res = compute_adf(ts_values)
                adf_df = pd.DataFrame({"Metric": adf_res.keys(), "Value": adf_res.values()})
                display(widgets.HTML("<b>ADF Stationarity Test</b>"))
                display(adf_df)

                # Seasonality
                seas = monthly_seasonality(ts_values, ts_dates)
                display(px.bar(seas, x="month", y="value", title="Monthly Seasonality"))

            # ADD TABS
            tab_titles.append(f"{sel['commodity']} ({sel['country']})")
            tab_contents.append(out)

            tab_titles.append(f"{sel['commodity']} ({sel['country']}) - TS")
            tab_contents.append(ts_out)

        # Show final tabs
        if tab_contents:
            tabs = widgets.Tab(children=tab_contents)
            for i, t in enumerate(tab_titles):
                tabs.set_title(i, t[:25])
            display(tabs)


#==================
# Add / Remove / Clear Buttons
#==================
widget_set = create_widget_set("")
selection_list = selections

btn_add = widgets.Button(description="➕ Add Selection", button_style='success')
btn_remove = widgets.Button(description="❌ Remove Selected", button_style='warning')
btn_clear = widgets.Button(description="🗑 Clear All", button_style='danger')

remove_dropdown = widgets.Dropdown(options=[], description="Remove:", layout=widgets.Layout(width="450px"))

def refresh_remove_dropdown():
    if not selection_list:
        remove_dropdown.options = []
        remove_dropdown.value = None
        return
    labels = [
        f"{i+1}: {s['commodity_type']} | {s['commodity']} | {s['price_type']} | {s['country']} | {s['market']} | {s['source']}"
        for i, s in enumerate(selection_list)
    ]
    remove_dropdown.options = labels
    remove_dropdown.value = labels[0]

def add_selection(_):
    vals = {k: w.value for k, w in widget_set.items()}
    vals["label"] = (
        f"{vals['commodity_type']} | {vals['commodity']} | {vals['price_type']} | "
        f"{vals['source']} | {vals['country']} | {vals['market']}"
    )
    selection_list.append(vals)
    refresh_remove_dropdown()
    update_plot()

def remove_selected(_):
    if not selection_list or not remove_dropdown.value:
        return
    idx = remove_dropdown.options.index(remove_dropdown.value)
    del selection_list[idx]
    refresh_remove_dropdown()
    update_plot()

def clear_all(_):
    selection_list.clear()
    refresh_remove_dropdown()
    update_plot()

btn_add.on_click(add_selection)
btn_remove.on_click(remove_selected)
btn_clear.on_click(clear_all)

#==================
# Display
#==================
ui = widgets.VBox(
    list(widget_set.values()) +
    [widgets.HBox([btn_add, btn_remove, btn_clear]), remove_dropdown, output_plot]
)
display(ui)